# 12 - Funder and installer high-cardinality screen

This notebook presents the bounded fixed-model organisation-feature screen. It performs no model refits: run `../scripts/run_high_cardinality_screen.py` first to recreate the ignored runtime evidence. The labelled local test, competition predictions and oversampling artefacts remain outside this analysis.

In [1]:
from pathlib import Path
import sys

import pandas as pd

STAGE_DIR = Path.cwd().parent
PROJECT_DIR = STAGE_DIR.parent
SRC_DIR = STAGE_DIR / 'src'
RUNTIME_DIR = PROJECT_DIR / '.runtime' / 'high-cardinality-screen'
DATA_DIR = STAGE_DIR / 'data'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from high_cardinality_features import normalise_organisation

required = [
    RUNTIME_DIR / 'policy-register.csv',
    RUNTIME_DIR / 'frozen-summary.csv',
    RUNTIME_DIR / 'lga-grouped-summary.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        f'Run the high-cardinality screen first; missing={missing!r}'
    )

print(f'Runtime evidence: {RUNTIME_DIR}')

Runtime evidence: C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\.runtime\high-cardinality-screen


## Audit boundary

Normalisation changes case and repeated whitespace only. It does not fuzzy-match organisation aliases or collapse literal sentinels. Learned rare and frequency mappings are fitted inside the current training partition.

In [2]:
training = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')
audit_rows = []
normalised_training = {}
for feature in ('funder', 'installer'):
    train_values = normalise_organisation(training[feature])
    test_values = normalise_organisation(competition[feature])
    normalised_training[feature] = train_values
    counts = train_values.value_counts()
    audit_rows.append({
        'feature': feature,
        'raw_levels': int(training[feature].nunique(dropna=False)),
        'normalised_levels': int(counts.size),
        'singleton_levels': int(counts.eq(1).sum()),
        'rows_in_levels_below_20': int(train_values.map(counts).lt(20).sum()),
        'competition_unseen_rows': int((~test_values.isin(counts.index)).sum()),
        'competition_unseen_share': float((~test_values.isin(counts.index)).mean()),
    })
audit = pd.DataFrame(audit_rows).set_index('feature')
audit['competition_unseen_share'] = audit['competition_unseen_share'].map(
    lambda value: f'{value:.3%}'
)
display(audit)
print(
    'Exact normalised funder/installer equality: '
    f"{normalised_training['funder'].eq(normalised_training['installer']).mean():.1%}"
)

,raw_levels,normalised_levels,singleton_levels,rows_in_levels_below_20,competition_unseen_rows,competition_unseen_share
feature,,,,,,
funder,1897,1897,974,4973,254,1.710%
installer,2146,1919,963,5041,233,1.569%


Exact normalised funder/installer equality: 37.9%


## Predeclared representations

The screen introduces funder and installer separately and together through a 20-row rare group, training-fold prevalence, or both. One bounded joint sensitivity raises the rare threshold to 50. Frequency maps unseen validation values to zero; no target values participate.

In [3]:
policies = pd.read_csv(RUNTIME_DIR / 'policy-register.csv').set_index('policy')
policies

,label,engineered_features,categorical_organisations,frequency_organisations,rare_category_minimum,rationale
policy,,,,,,
baseline,Accepted feature policy,29,NaN,NaN,20,No funder or installer feature
funder_rare20,Funder rare-grouped categorical,30,funder,NaN,20,Normalised funder with fold-fitted 20-row rare...
installer_rare20,Installer rare-grouped categorical,30,installer,NaN,20,Normalised installer with fold-fitted 20-row r...
both_rare20,Funder and installer rare-grouped categoricals,31,"funder, installer",NaN,20,Add both normalised organisation identities se...
funder_frequency,Funder frequency,30,NaN,funder,20,Fold-fitted funder prevalence with unseen valu...
installer_frequency,Installer frequency,30,NaN,installer,20,Fold-fitted installer prevalence with unseen v...
both_frequency,Funder and installer frequencies,31,NaN,"funder, installer",20,Add both fold-fitted organisation prevalence f...
funder_rare20_frequency,Funder rare-grouped categorical plus frequency,31,funder,funder,20,Retain funder identity and expose its training...
installer_rare20_frequency,Installer rare-grouped categorical plus frequency,31,installer,installer,20,Retain installer identity and expose its train...


## Frozen-fold screen

Every policy uses the unchanged 55% child-weight-1 depth-8 XGBoost and 45% Random Forest vote. The primary gate requires at least +0.10 percentage points, three fold wins, no fold below -0.25 points and no repair-recall loss beyond two points.

In [4]:
frozen = pd.read_csv(RUNTIME_DIR / 'frozen-summary.csv').set_index('policy')
frozen_display = frozen.loc[:, [
    'label', 'mean_accuracy', 'accuracy_change', 'fold_wins',
    'worst_fold_change', 'repair_recall',
    'transformed_features_fold_1', 'passes_gate',
]].copy()
for column in (
    'mean_accuracy', 'accuracy_change', 'worst_fold_change', 'repair_recall'
):
    frozen_display[column] = frozen_display[column].map(
        lambda value: f'{value:.3%}'
    )
frozen_display

,label,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,transformed_features_fold_1,passes_gate
policy,,,,,,,,
funder_frequency,Funder frequency,81.656%,0.032%,2,-0.105%,34.917%,302,False
baseline,Accepted feature policy,81.625%,0.000%,0,0.000%,34.859%,301,False
funder_rare20,Funder rare-grouped categorical,81.599%,-0.025%,1,-0.147%,34.453%,499,False
funder_rare20_frequency,Funder rare-grouped categorical plus frequency,81.589%,-0.036%,1,-0.074%,34.396%,500,False
both_frequency,Funder and installer frequencies,81.574%,-0.051%,2,-0.263%,34.453%,303,False
installer_rare20,Installer rare-grouped categorical,81.538%,-0.086%,0,-0.231%,34.511%,476,False
both_rare20_frequency,Both rare-grouped categoricals plus frequencies,81.532%,-0.093%,1,-0.168%,33.961%,676,False
both_rare20,Funder and installer rare-grouped categoricals,81.524%,-0.101%,0,-0.137%,34.425%,674,False
both_rare50_frequency,Both 50-row rare groups plus frequencies,81.507%,-0.118%,0,-0.168%,34.077%,503,False


Funder frequency is the only challenger above baseline: 81.656% versus 81.625%. The 0.032-point difference is below the gate, wins only two folds and is not improved by retaining supported funder identities. Installer and joint representations are weaker.

## LGA-disjoint sensitivity

The best frozen-fold challenger receives one bounded LGA-disjoint check because organisation prevalence may proxy geography. This is robustness evidence, not a second promotion opportunity.

In [5]:
grouped = pd.read_csv(RUNTIME_DIR / 'lga-grouped-summary.csv').set_index('policy')
grouped_display = grouped.loc[:, [
    'label', 'mean_accuracy', 'accuracy_change', 'fold_wins',
    'worst_fold_change', 'repair_recall',
    'xgboost_accuracy', 'random_forest_accuracy', 'passes_gate',
]].copy()
for column in (
    'mean_accuracy', 'accuracy_change', 'worst_fold_change',
    'repair_recall', 'xgboost_accuracy', 'random_forest_accuracy',
):
    grouped_display[column] = grouped_display[column].map(
        lambda value: f'{value:.3%}'
    )
grouped_display

,label,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,xgboost_accuracy,random_forest_accuracy,passes_gate
policy,,,,,,,,,
funder_frequency,Funder frequency,72.350%,0.250%,3,-0.274%,2.959%,71.736%,72.766%,False
baseline,Accepted feature policy,72.101%,0.000%,0,0.000%,3.497%,71.694%,72.192%,False


## Decision

Retain the accepted 29-feature policy. Funder frequency improves the LGA-disjoint mean by 0.250 points, but the primary frozen-fold gain is too small and unstable; the grouped result also loses 0.274 points in its worst fold and reduces repair recall from 3.50% to 2.96%. Stop tuning organisation thresholds on these folds. Move the next data-focused loop to numeric state-plus-magnitude treatments for `amount_tsh`, `population`, `gps_height` and `num_private`.